In [1]:
import pandas as pd
import json
import joblib
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
import os

with open('artifacts/feature_list.json', 'r') as f:
    features = json.load(f)

train_df = pd.read_csv('artifacts/train_engineered.csv')
val_df = pd.read_csv('artifacts/val_engineered.csv')
test_df = pd.read_csv('artifacts/test_engineered.csv')

X_train, y_train = train_df[features], train_df['is_late']
X_val, y_val = val_df[features], val_df['is_late']
X_test, y_test = test_df[features], test_df['is_late']

baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, y_train)
baseline_auc = roc_auc_score(y_val, baseline.predict_proba(X_val)[:, 1])
print(f"Baseline ROC-AUC Score: {baseline_auc:.4f}")

print("\nTraining Random Forest Model...")
model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
model.fit(X_train, y_train)

val_auc = roc_auc_score(y_val, model.predict_proba(X_val)[:, 1])
print(f"Validation ROC-AUC Score (Our Model): {val_auc:.4f}")

Baseline ROC-AUC Score: 0.5000

Training Random Forest Model...
Validation ROC-AUC Score (Our Model): 0.5821


In [2]:
test_auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
print(f"Final Test ROC-AUC Score: {test_auc:.4f}")

os.makedirs('artifacts', exist_ok=True)
joblib.dump(model, 'artifacts/final_model.pkl')

results_summary = {
    "baseline_roc_auc": float(baseline_auc),
    "validation_roc_auc": float(val_auc),
    "test_roc_auc": float(test_auc),
    "model_type": "RandomForestClassifier"
}

with open('artifacts/results_summary.json', 'w') as f:
    json.dump(results_summary, f, indent=4)

print("\nFinal model and results summary saved successfully in artifacts!")

Final Test ROC-AUC Score: 0.5767

Final model and results summary saved successfully in artifacts!
